In [1]:
%pip install pandas numpy scikit-learn joblib pytest

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import sklearn
import joblib
import pytest
import os
import warnings

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

warnings.filterwarnings("ignore")

print("All libraries imported successfully!")

All libraries imported successfully!


In [3]:
data = pd.read_csv("kidney_disease.csv")

print("Dataset loaded successfully!")
print("Dataset Shape:", data.shape)

display(data.head())

Dataset loaded successfully!
Dataset Shape: (400, 26)


,id,age,bp,sg,al,su,rbc,pc,pcc,ba,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,...,44,7800,5.2,yes,yes,no,good,no,no,ckd
1,1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,...,38,6000,NaN,no,no,no,good,no,no,ckd
2,2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,...,31,7500,NaN,no,yes,no,poor,no,yes,ckd
3,3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,...,32,6700,3.9,yes,no,no,poor,yes,yes,ckd
4,4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,...,35,7300,4.6,no,no,no,good,no,no,ckd


In [4]:
print("Dataset Information:")
data.info()

print("\nMissing Values:")
print(data.isnull().sum())

print("\nTarget Distribution:")
print(data["classification"].value_counts())

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 26 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              400 non-null    int64  
 1   age             391 non-null    float64
 2   bp              388 non-null    float64
 3   sg              353 non-null    float64
 4   al              354 non-null    float64
 5   su              351 non-null    float64
 6   rbc             248 non-null    object 
 7   pc              335 non-null    object 
 8   pcc             396 non-null    object 
 9   ba              396 non-null    object 
 10  bgr             356 non-null    float64
 11  bu              381 non-null    float64
 12  sc              383 non-null    float64
 13  sod             313 non-null    float64
 14  pot             312 non-null    float64
 15  hemo            348 non-null    float64
 16  pcv             330 non-null    object 
 17  wc            

In [5]:
data.columns = data.columns.str.strip().str.lower()

data = data.loc[
    :, ~data.columns.str.contains("^unnamed")
]

data = data.drop(columns=["id"], errors="ignore")

for col in data.select_dtypes(include="object").columns:
    data[col] = data[col].str.strip().str.lower()

numeric_columns = [
    "age", "bp", "sg", "al", "su", "bgr",
    "bu", "sc", "sod", "pot", "hemo",
    "pcv", "wc", "rc"
]

for col in numeric_columns:
    if col in data.columns:
        data[col] = pd.to_numeric(data[col], errors="coerce")

data["classification"] = (
    data["classification"]
    .astype("string")
    .str.strip()
    .str.lower()
)

data = data.dropna(subset=["classification"])

print("Dataset cleaned successfully!")
print("Cleaned Dataset Shape:", data.shape)

Dataset cleaned successfully!
Cleaned Dataset Shape: (400, 25)


In [6]:
X = data.drop(columns=["classification"])
y = data["classification"]

print("Input Features Shape:", X.shape)
print("Target Shape:", y.shape)

print("\nTarget Classes:")
print(y.unique())

Input Features Shape: (400, 24)
Target Shape: (400,)

Target Classes:
<StringArray>
['ckd', 'notckd']
Length: 2, dtype: string


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Records:", X_train.shape[0])
print("Testing Records:", X_test.shape[0])

Training Records: 320
Testing Records: 80


In [8]:
numerical_columns = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_columns = X_train.select_dtypes(
    include=["object", "string"]
).columns.tolist()

print("Numerical Columns:", numerical_columns)
print("\nCategorical Columns:", categorical_columns)

Numerical Columns: ['age', 'bp', 'sg', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc']

Categorical Columns: ['rbc', 'pc', 'pcc', 'ba', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane']


In [9]:
numerical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numerical_pipeline, numerical_columns),
    ("cat", categorical_pipeline, categorical_columns)
])

print("Preprocessing pipeline created successfully!")

Preprocessing pipeline created successfully!


In [10]:
ml_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

print("Machine Learning Pipeline Created!")

Machine Learning Pipeline Created!


In [11]:
ml_pipeline.fit(X_train, y_train)

print("Model training completed successfully!")

Model training completed successfully!


In [12]:
y_pred = ml_pipeline.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Model Accuracy: 0.975

Classification Report:
              precision    recall  f1-score   support

         ckd       0.96      1.00      0.98        50
      notckd       1.00      0.93      0.97        30

    accuracy                           0.97        80
   macro avg       0.98      0.97      0.97        80
weighted avg       0.98      0.97      0.97        80



In [13]:
joblib.dump(ml_pipeline, "kidney_cicd_model.pkl")

print("Model saved successfully!")

Model saved successfully!


In [14]:
loaded_model = joblib.load("kidney_cicd_model.pkl")

sample_input = X_test.iloc[[0]]

sample_prediction = loaded_model.predict(sample_input)

print("Sample Prediction:", sample_prediction[0])
print("Actual Classification:", y_test.iloc[0])
print("Model loaded and tested successfully!")

Sample Prediction: ckd
Actual Classification: ckd
Model loaded and tested successfully!


In [15]:
%%writefile train_model.py

import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier


def train_model():

    data = pd.read_csv("kidney_disease.csv")

    data.columns = data.columns.str.strip().str.lower()
    data = data.loc[:, ~data.columns.str.contains("^unnamed")]
    data = data.drop(columns=["id"], errors="ignore")

    for col in data.select_dtypes(include="object").columns:
        data[col] = data[col].str.strip().str.lower()

    numeric_columns = [
        "age", "bp", "sg", "al", "su", "bgr",
        "bu", "sc", "sod", "pot", "hemo",
        "pcv", "wc", "rc"
    ]

    for col in numeric_columns:
        if col in data.columns:
            data[col] = pd.to_numeric(data[col], errors="coerce")

    data["classification"] = (
        data["classification"].astype("string").str.strip().str.lower()
    )

    data = data.dropna(subset=["classification"])

    X = data.drop(columns=["classification"])
    y = data["classification"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    numerical_columns = X_train.select_dtypes(
        include=["int64", "float64"]
    ).columns.tolist()

    categorical_columns = X_train.select_dtypes(
        include=["object", "string"]
    ).columns.tolist()

    numerical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("num", numerical_pipeline, numerical_columns),
        ("cat", categorical_pipeline, categorical_columns)
    ])

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=100,
            random_state=42
        ))
    ])

    model.fit(X_train, y_train)

    joblib.dump(model, "kidney_cicd_model.pkl")

    print("Model trained and saved successfully!")

    return model


if __name__ == "__main__":
    train_model()

Writing train_model.py


In [16]:
%%writefile test_model.py

import pandas as pd
import joblib
from pathlib import Path

MODEL_PATH = "kidney_cicd_model.pkl"


def get_sample_input():

    data = pd.read_csv("kidney_disease.csv")

    data.columns = data.columns.str.strip().str.lower()
    data = data.loc[:, ~data.columns.str.contains("^unnamed")]
    data = data.drop(columns=["id"], errors="ignore")

    for col in data.select_dtypes(include="object").columns:
        data[col] = data[col].str.strip().str.lower()

    numeric_columns = [
        "age", "bp", "sg", "al", "su", "bgr",
        "bu", "sc", "sod", "pot", "hemo",
        "pcv", "wc", "rc"
    ]

    for col in numeric_columns:
        if col in data.columns:
            data[col] = pd.to_numeric(data[col], errors="coerce")

    data = data.dropna(subset=["classification"])

    X = data.drop(columns=["classification"])

    return X.iloc[[0]]


def test_model_file_exists():
    assert Path(MODEL_PATH).exists()


def test_model_prediction():

    model = joblib.load(MODEL_PATH)

    sample = get_sample_input()

    prediction = model.predict(sample)

    assert len(prediction) == 1
    assert prediction[0] in model.classes_

Writing test_model.py


In [17]:
%%writefile requirements.txt

pandas
numpy
scikit-learn
joblib
pytest

Writing requirements.txt


In [18]:
required_files = [
    "kidney_disease.csv",
    "Experiment_09.ipynb",
    "kidney_cicd_model.pkl",
    "train_model.py",
    "test_model.py",
    "requirements.txt"
]

for file in required_files:
    print(file, "Available" if os.path.exists(file) else "Missing")

kidney_disease.csv Available
Experiment_09.ipynb Available
kidney_cicd_model.pkl Available
train_model.py Available
test_model.py Available
requirements.txt Available


In [19]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "train_model.py"],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
else:
    print("Training script executed successfully!")

Model trained and saved successfully!

Training script executed successfully!


In [20]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "test_model.py", "-v"],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
else:
    print("All automated tests passed!")

============================= test session starts =============================
platform win32 -- Python 3.13.9, pytest-8.4.2, pluggy-1.5.0 -- C:\ProgramData\anaconda3\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\bhava\Experiment_09_CICD_Pipeline
plugins: hydra-core-1.3.6, typeguard-4.6.0, anyio-4.10.0
collecting ... collected 2 items

test_model.py::test_model_file_exists PASSED                             [ 50%]
test_model.py::test_model_prediction PASSED                              [100%]

============================== 2 passed in 2.16s ==============================

All automated tests passed!


In [21]:
print("Model File Exists:", os.path.exists("kidney_cicd_model.pkl"))

model = joblib.load("kidney_cicd_model.pkl")

print("Model Loaded Successfully!")
print("Model Classes:", model.classes_)

Model File Exists: True
Model Loaded Successfully!
Model Classes: ['ckd' 'notckd']


In [22]:
os.makedirs(".github/workflows", exist_ok=True)

print("GitHub Actions workflow directory created!")

GitHub Actions workflow directory created!


In [23]:
%%writefile .github/workflows/ci.yml

name: Kidney Disease ML CI-CD Pipeline

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  build-and-test:

    runs-on: ubuntu-latest

    steps:
      - name: Checkout Repository
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Install Dependencies
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt

      - name: Train Machine Learning Model
        run: python train_model.py

      - name: Run Automated Tests
        run: pytest -v

      - name: Publish Model Artifact
        uses: actions/upload-artifact@v4
        with:
          name: kidney-disease-trained-model
          path: kidney_cicd_model.pkl

Writing .github/workflows/ci.yml


In [24]:
workflow_path = ".github/workflows/ci.yml"

print("Workflow File Available:", os.path.exists(workflow_path))

with open(workflow_path, "r") as file:
    print(file.read())

Workflow File Available: True

name: Kidney Disease ML CI-CD Pipeline

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  build-and-test:

    runs-on: ubuntu-latest

    steps:
      - name: Checkout Repository
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Install Dependencies
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt

      - name: Train Machine Learning Model
        run: python train_model.py

      - name: Run Automated Tests
        run: pytest -v

      - name: Publish Model Artifact
        uses: actions/upload-artifact@v4
        with:
          name: kidney-disease-trained-model
          path: kidney_cicd_model.pkl



In [25]:
for root, dirs, files in os.walk("."):
    dirs[:] = [d for d in dirs if d not in [".ipynb_checkpoints", "__pycache__", ".git"]]
    level = root.count(os.sep)
    print("    " * level + os.path.basename(root) + "/")

    for file in files:
        print("    " * (level + 1) + file)

./
    Experiment_09.ipynb
    kidney_cicd_model.pkl
    kidney_disease.csv
    requirements.txt
    test_model.py
    train_model.py
    .github/
        workflows/
            ci.yml
    .pytest_cache/
        .gitignore
        CACHEDIR.TAG
        README.md
        v/
            cache/
                nodeids


In [26]:
%%writefile .gitignore

.ipynb_checkpoints/
__pycache__/
*.pyc
kidney_cicd_model.pkl

Writing .gitignore
